In [ ]:
import os
import pandas as pd
import seaborn as sns

from pathlib import Path


import ipywidgets as widgets


from IPython.display import display


from scripts.config import model_paths, paths
from scripts.load_model import load_model

from scripts.causal_plots import (
    plot_r2_scores,
    plot_total_coefficients,
    plot_falsification_hist
)
%load_ext autoreload
%autoreload 2

## Load model run

In [ ]:
models_folder = Path("../" + model_paths)
print(f"Models folder: {models_folder}")


subfolder_names = [p.name for p in models_folder.iterdir() if p.is_dir()]


# Create dropdown for folder selection


folder_dropdown = widgets.Dropdown(
    options=subfolder_names,
    description="Folder:",
    disabled=False,
)


# Display the dropdown


display(folder_dropdown)


load_button = widgets.Button(description="Load Folder")


output = widgets.Output()


data_selected = pd.read_csv(paths["data_selected"]).set_index("timestamp")


data_selected.index = pd.to_datetime(data_selected.index, utc="utc")


# Shared state dictionary


state = {"model": None}


def on_load_clicked(b):

    with output:
        output.clear_output()

        selected_folder = os.path.join(models_folder, folder_dropdown.value)

        print(f"You selected: {selected_folder}")
        model = load_model(
            model_folder=os.path.join(models_folder, selected_folder),
            data=data_selected,
        )

        state["model"] = model

        print(f"Model run {folder_dropdown.value} has been loaded...")


load_button.on_click(on_load_clicked)


display(load_button, output)

## Print edges of graph

In [ ]:
for i, graph in enumerate(state["model"].keys()):
    print(f"Graph: {graph}")
    print(state["model"][graph]["edges"])

## Falsification and evaluation

In [ ]:
# print(state["model"]["graph14"]["r2_values"])
for i, graph in enumerate(state["model"].keys()):
    print(f"Graph: {graph}")
    if "r2_scores" not in state["model"][graph]:
        print("No R2 scores available for this graph.")
        continue
    plot_r2_scores(
        r2_scores=state["model"][graph]["r2_scores"],
        fig_dir=state["model"][graph]["fig_dir"],
        color=sns.color_palette("colorblind")[i],
    )

In [ ]:
for graph in state["model"].keys():
    print(f"Graph: {graph}")
    if "falsification" not in state["model"][graph]:
        print("No falsification data available for this graph.")
        continue
    print(f"{state['model'][graph]['falsification']}")
    plot_falsification_hist(
        falsification=state["model"][graph]["falsification"],
        fig_dir=state["model"][graph]["fig_dir"],
    )

## Coefficients

In [ ]:
for i, graph in enumerate(state["model"].keys()):
    print(f"Graph: {graph}")
    target = state["model"][graph]["target"]
    if "coefficients" not in state["model"][graph]:
        print("No coefficients available for this graph.")
        continue
    if "price" in target:
        unit = "EUR/MWh"
        exports = False
    elif "export" in target:
        unit = "GW"
        exports = True
    else:
        print(f"Unknown target variable {target}, cannot determine unit.")
        continue
    plot_total_coefficients(
        data_by_time=state["model"][graph]["data_by_time"],
        coefficients=state["model"][graph]["coefficients"],
        fig_dir=state["model"][graph]["fig_dir"],
        color=sns.color_palette("colorblind")[i],
        unit=unit,
        exports=exports,
    )